In [1]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../../db.sqlite3")

ratings = pd.read_sql_query("""
    SELECT u.id AS userId, u.username, m.movie_id AS movieId, r.score AS rating
    FROM movies_rating r
    JOIN movies_movie m ON r.movie_id = m.id
    JOIN auth_user u ON r.user_id = u.id
""", conn)

movies = pd.read_sql_query("SELECT movie_id AS movieId, title, genres FROM movies_movie", conn)
conn.close()

print(len(ratings), "ratings,", ratings["userId"].nunique(), "users")
ratings.head()

100861 ratings, 611 users


,userId,username,movieId,rating
0,2,ml_user_1,1,4.0
1,2,ml_user_1,3,4.0
2,2,ml_user_1,6,4.0
3,2,ml_user_1,47,5.0
4,2,ml_user_1,50,5.0


In [6]:
MY_USERNAME = "rahul_test"   # change this to your signup username

my_id = ratings.loc[ratings["username"] == MY_USERNAME, "userId"].iloc[0]
print("My user id:", my_id)

mine = ratings[ratings["userId"] == my_id].merge(movies, on="movieId")
mine[["title", "rating"]].sort_values("rating", ascending=False)

My user id: 612


,title,rating
3,"Shawshank Redemption, The (1994)",5.0
24,Up (2009),4.5
2,Toy Story (1995),4.5
20,Batman Forever (1995),4.5
18,Reservoir Dogs (1992),4.5
16,Die Hard (1988),4.5
23,Kill Bill: Vol. 2 (2004),4.0
22,Catch Me If You Can (2002),4.0
7,Batman (1989),4.0
8,Seven (a.k.a. Se7en) (1995),4.0


In [7]:
pivot = ratings.pivot_table(index="userId", columns="movieId", values="rating")
print("Matrix shape:", pivot.shape)
print(f"Filled: {pivot.notna().sum().sum() / pivot.size:.2%}")

Matrix shape: (611, 9724)
Filled: 1.70%


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# keep movies with at least 20 ratings (enough evidence, and keeps memory small)
counts = pivot.notna().sum()
pivot_f = pivot.loc[:, counts >= 20]

# subtract each user's average, so generous raters and harsh raters become comparable
centered = pivot_f.sub(pivot_f.mean(axis=1), axis=0).fillna(0)

item_sim = pd.DataFrame(
    cosine_similarity(centered.T),
    index=pivot_f.columns, columns=pivot_f.columns,
)
print(item_sim.shape)

(1297, 1297)


In [9]:
title_to_id = movies.set_index("title")["movieId"]
id_to_title = movies.set_index("movieId")["title"]

def similar_by_ratings(title, n=10):
    mid = title_to_id[title]
    s = item_sim[mid].drop(mid).sort_values(ascending=False).head(n)
    return pd.DataFrame({"title": s.index.map(id_to_title), "similarity": s.values.round(3)})

similar_by_ratings("Toy Story (1995)")

,title,similarity
0,Toy Story 2 (1999),0.350
1,Aladdin (1992),0.281
2,Toy Story 3 (2010),0.278
3,Wallace & Gromit: The Wrong Trousers (1993),0.263
4,"Incredibles, The (2004)",0.224
5,Finding Nemo (2003),0.219
6,Blazing Saddles (1974),0.210
7,Back to the Future (1985),0.210
8,Babe (1995),0.199
9,"Lion King, The (1994)",0.186


In [10]:
me = pivot_f.loc[my_id].dropna()          # movies I rated (that are in the matrix)
me_mean = me.mean()

w = item_sim[me.index]                    # similarity of every movie to the ones I rated
score = w.dot(me - me_mean) / w.abs().sum(axis=1) + me_mean

score = score.drop(me.index, errors="ignore").sort_values(ascending=False).head(10)
pd.DataFrame({"title": score.index.map(id_to_title), "predicted": score.values.round(2)})

,title,predicted
0,Die Hard: With a Vengeance (1995),4.30
1,Gosford Park (2001),4.25
2,Four Weddings and a Funeral (1994),4.21
3,"Quick and the Dead, The (1995)",4.18
4,Get Shorty (1995),4.09
5,"Crying Game, The (1992)",4.06
6,Don Juan DeMarco (1995),4.06
7,"Remains of the Day, The (1993)",4.06
8,Sleeper (1973),4.06
9,Babe: Pig in the City (1998),4.05


In [11]:
from surprise import Dataset, Reader, BaselineOnly, SVD, accuracy
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[["userId", "movieId", "rating"]], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

baseline = BaselineOnly().fit(trainset)
print("Baseline RMSE:", accuracy.rmse(baseline.test(testset)))

Estimating biases using als...
RMSE: 0.8787
Baseline RMSE: 0.8787398061558069


In [12]:
svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)

preds = svd.test(testset)
print("SVD RMSE:", accuracy.rmse(preds))
print("SVD MAE: ", accuracy.mae(preds))

RMSE: 0.8774
SVD RMSE: 0.8773985863312617
MAE:  0.6746
SVD MAE:  0.6745646625094416


In [13]:
# retrain on ALL data (including your ratings) for the final model
full = data.build_full_trainset()
final_svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
final_svd.fit(full)

rated_ids = set(ratings.loc[ratings["userId"] == my_id, "movieId"])
candidates = [m for m in movies["movieId"] if m not in rated_ids]

scored = [(m, final_svd.predict(my_id, m).est) for m in candidates]
scored.sort(key=lambda x: x[1], reverse=True)

top = pd.DataFrame(scored[:10], columns=["movieId", "predicted"]).merge(movies, on="movieId")
top[["title", "genres", "predicted"]]

,title,genres,predicted
0,Lawrence of Arabia (1962),Adventure|Drama|War,4.127798
1,"Great Escape, The (1963)",Action|Adventure|Drama|War,4.110511
2,Wallace & Gromit: The Best of Aardman Animatio...,Adventure|Animation|Comedy,4.101277
3,"Streetcar Named Desire, A (1951)",Drama,4.099889
4,"Philadelphia Story, The (1940)",Comedy|Drama|Romance,4.095342
5,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.094164
6,Apocalypse Now (1979),Action|Drama|War,4.088906
7,Casablanca (1942),Drama|Romance,4.082037
8,There Will Be Blood (2007),Drama|Western,4.069696
9,"Departed, The (2006)",Crime|Drama|Thriller,4.064088


In [14]:
movie_counts = ratings.groupby("movieId").size()

def recommend_svd(model, user_id, n=10, min_ratings=30):
    rated = set(ratings.loc[ratings["userId"] == user_id, "movieId"])
    popular = set(movie_counts[movie_counts >= min_ratings].index)
    candidates = [m for m in popular if m not in rated]

    scored = [(m, model.predict(user_id, m).est) for m in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)

    out = pd.DataFrame(scored[:n], columns=["movieId", "predicted"]).merge(movies, on="movieId")
    out["num_ratings"] = out["movieId"].map(movie_counts)
    return out[["title", "genres", "predicted", "num_ratings"]]

recommend_svd(final_svd, my_id)

,title,genres,predicted,num_ratings
0,Lawrence of Arabia (1962),Adventure|Drama|War,4.127798,45
1,"Great Escape, The (1963)",Action|Adventure|Drama|War,4.110511,43
2,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.094164,142
3,Apocalypse Now (1979),Action|Drama|War,4.088906,107
4,Casablanca (1942),Drama|Romance,4.082037,100
5,"Departed, The (2006)",Crime|Drama|Thriller,4.064088,107
6,Chinatown (1974),Crime|Film-Noir|Mystery|Thriller,4.054145,59
7,Old Boy (2003),Mystery|Thriller,4.029994,39
8,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,3.997867,200
9,Wallace & Gromit: The Wrong Trousers (1993),Animation|Children|Comedy|Crime,3.994081,56


In [15]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=10, threshold=3.5):
    by_user = defaultdict(list)
    for p in predictions:
        by_user[p.uid].append((p.est, p.r_ui))

    precisions, recalls = [], []
    for uid, items in by_user.items():
        items.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum(true >= threshold for _, true in items)
        n_rec = sum(est >= threshold for est, _ in items[:k])
        n_hit = sum((true >= threshold) and (est >= threshold) for est, true in items[:k])
        precisions.append(n_hit / n_rec if n_rec else 0)
        recalls.append(n_hit / n_rel if n_rel else 0)

    return np.mean(precisions), np.mean(recalls)

p, r = precision_recall_at_k(baseline.test(testset))
print(f"Baseline  Precision@10: {p:.3f}  Recall@10: {r:.3f}")

p, r = precision_recall_at_k(svd.test(testset))
print(f"SVD       Precision@10: {p:.3f}  Recall@10: {r:.3f}")

Baseline  Precision@10: 0.742  Recall@10: 0.513
SVD       Precision@10: 0.743  Recall@10: 0.505


In [16]:
from surprise.model_selection import GridSearchCV

param_grid = {
    "n_factors": [20, 50, 100],
    "n_epochs": [20, 30],
    "lr_all": [0.005],
    "reg_all": [0.02, 0.1],
}

gs = GridSearchCV(SVD, param_grid, measures=["rmse"], cv=3, n_jobs=1)
gs.fit(data)

print("Best RMSE:", gs.best_score["rmse"])
print("Best params:", gs.best_params["rmse"])

Best RMSE: 0.8702567165178969
Best params: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}


In [17]:
import joblib, os

os.makedirs("../artifacts", exist_ok=True)

best = SVD(**gs.best_params["rmse"], random_state=42)
best.fit(data.build_full_trainset())

joblib.dump(best, "../artifacts/svd_v1.pkl")
print("Saved svd_v1.pkl")

Saved svd_v1.pkl
